# 🛠️ Notebook 02 — CDR Feature Engineering
## Weight of Evidence (WoE), IV Analysis & Feature Selection

This notebook focuses on feature engineering from raw CDR events including
WoE binning, Information Value ranking, and network topology feature extraction.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, sys, os
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore")

# Load dataset (notebook 01 has been run)
exec(open('../Users/Gugu Ncube/Desktop/Documents/Modelling/HandsetFinance.py').read())
FEATURE_COLS = [c for c in df.columns if c not in ["subscriber_id","latent_creditworthiness","default_flag"]]
print(f"Dataset loaded: {df.shape}")


## 📊 Weight of Evidence (WoE) & Information Value (IV)

In [ ]:
def compute_iv(series, target, n_bins=10):
    """Compute Information Value for a single feature."""
    df_tmp = pd.DataFrame({"feature": series, "target": target}).dropna()
    try:
        df_tmp["bin"] = pd.qcut(df_tmp["feature"], n_bins, duplicates="drop")
    except:
        df_tmp["bin"] = pd.cut(df_tmp["feature"], n_bins)

    total_bads  = (df_tmp["target"]==1).sum()
    total_goods = (df_tmp["target"]==0).sum()

    tbl = df_tmp.groupby("bin")["target"].agg(
        bad_cnt  = lambda x: (x==1).sum(),
        good_cnt = lambda x: (x==0).sum()
    )
    tbl["bad_pct"]  = (tbl["bad_cnt"]  / total_bads ).clip(1e-6)
    tbl["good_pct"] = (tbl["good_cnt"] / total_goods).clip(1e-6)
    tbl["woe"] = np.log(tbl["good_pct"] / tbl["bad_pct"])
    tbl["iv"]  = (tbl["good_pct"] - tbl["bad_pct"]) * tbl["woe"]
    return tbl["iv"].sum()

print("Computing IV for all features...")
iv_results = pd.DataFrame([
    {"Feature": feat, "IV": round(compute_iv(df[feat], df["default_flag"]), 4)}
    for feat in FEATURE_COLS
]).sort_values("IV", ascending=False).reset_index(drop=True)

iv_results["Strength"] = iv_results["IV"].apply(
    lambda iv: "Very Strong" if iv>0.5 else "Strong" if iv>0.3
    else "Medium" if iv>0.1 else "Weak" if iv>0.02 else "Useless"
)
print(iv_results.head(20).to_string(index=False))


In [ ]:
# ── IV Bar Chart ──────────────────────────────────────────────────────────
top20 = iv_results.head(20)
colours = {"Very Strong":"#1B4F72","Strong":"#2E86C1","Medium":"#F39C12",
           "Weak":"#95A5A6","Useless":"#ECF0F1"}

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(
    top20["Feature"][::-1],
    top20["IV"][::-1],
    color=[colours[s] for s in top20["Strength"][::-1]],
    alpha=0.87, edgecolor="white"
)
ax.axvline(0.30, color="#2E86C1", ls="--", lw=1.2, label="Strong (0.30)")
ax.axvline(0.10, color="#F39C12", ls="--", lw=1.2, label="Medium (0.10)")
ax.set(title="Information Value — Top 20 CDR Features",
       xlabel="Information Value (IV)", ylabel="")
ax.legend(fontsize=9)
for bar, val in zip(bars, top20["IV"][::-1]):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2, f"{val:.3f}",
            va="center", fontsize=8.5, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/figures/08_iv_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# Feature selection: keep IV > 0.02
selected_features = iv_results[iv_results["IV"] > 0.02]["Feature"].tolist()
print(f"\nSelected {len(selected_features)} features with IV > 0.02 (dropped {len(FEATURE_COLS)-len(selected_features)})")


## 🌐 Network Topology Feature Analysis

In [ ]:
# Analyse network features' relationship to default
network_feats = ["ego_network_size","network_clustering_coeff",
                 "betweenness_centrality","social_diversity_score"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, feat in zip(axes, network_feats):
    good = df.loc[df["default_flag"]==0, feat]
    bad  = df.loc[df["default_flag"]==1, feat]

    ax.hist(good, bins=40, alpha=0.6, color="#27AE60", density=True, label="Non-default")
    ax.hist(bad,  bins=40, alpha=0.6, color="#C0392B", density=True, label="Default")
    ax.set_title(feat.replace("_"," ").title())
    ax.legend(fontsize=9)

    # T-test annotation
    from scipy.stats import ttest_ind
    t, p = ttest_ind(good, bad)
    ax.text(0.98, 0.95, f"p={p:.2e}", transform=ax.transAxes,
            ha="right", fontsize=8.5, color="#1B4F72")

fig.suptitle("Network Topology Features — Default vs Non-default", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/figures/09_network_features.png", dpi=150, bbox_inches="tight")
plt.show()


## ✅ Feature Engineering Summary

In [ ]:
print("Feature Engineering Summary")
print("=" * 50)
print(f"Total CDR features: {len(FEATURE_COLS)}")
print(f"Selected (IV > 0.02): {len(selected_features)}")
print(f"\nTop 10 by IV:")
for _, row in iv_results.head(10).iterrows():
    print(f"  {row['Feature']:<40} IV={row['IV']:.4f} ({row['Strength']})")
